In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import warnings
warnings.filterwarnings('ignore')
import re
from bs4 import BeautifulSoup
import time

In [119]:
df=pd.read_csv(r'Uncleaned_oyo_rooms.csv')

In [120]:
df.shape

(1791, 11)

In [121]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1791 entries, 0 to 1790
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Unnamed: 0        1791 non-null   int64 
 1   Hotel_Name        1791 non-null   object
 2   Location          1791 non-null   object
 3   Ratings           1791 non-null   object
 4   Final_Price       1791 non-null   object
 5   Original_Price    1791 non-null   object
 6   Discount          1791 non-null   object
 7   Taxes             1791 non-null   object
 8   Wizard_Member     1791 non-null   object
 9   Amenities         1791 non-null   object
 10  Company_Serviced  1791 non-null   object
dtypes: int64(1), object(10)
memory usage: 154.0+ KB


In [122]:
df.head()

,Unnamed: 0,Hotel_Name,Location,Ratings,Final_Price,Original_Price,Discount,Taxes,Wizard_Member,Amenities,Company_Serviced
0,0,Townhouse Narayanaguda Metro Station Formerly ...,"Near Old MLA Quarters, Himayathnagar, Hyderabad",4.3 (774 Ratings)·Very Good,₹1730,₹7095,71% off,+ ₹257 taxes & fees · per room per night,Yes,Parking facility Dining area Free Wifi + 14...,Yes
1,1,Townhouse Kothapet Formerly Surabhi Elite,"Kothapet, Hyderabad",4.5 (320 Ratings)·Excellent,₹1363,₹7695,79% off,+ ₹239 taxes & fees · per room per night,Yes,Free Wifi Power backup Parking + 10 more,Yes
2,2,Townhouse Oak Madhapur Nera Regency,"Madhapur, Hyderabad",4.3 (1793 Ratings)·Very Good,₹1605,₹6498,72% off,+ ₹169 taxes & fees · per room per night,Yes,Dining area Power backup Parking + 29 more,Yes
3,3,Hotel O by OYO Saraswathi Residency Near SR Na...,"Near Nalanda School, S.R. Nagar, Hyderabad",4.3 (6 Ratings)·Very Good,₹1332,₹5381,71% off,+ ₹175 taxes & fees · per room per night,No,Elevator Free Wifi Geyser + 8 more,No
4,4,Super Townhouse OAK Hotel Belsons Taj Mahal,"Near Swapnalok Complex, Patny Centre, Hyderabad",4.6 (82 Ratings)·Excellent,₹1963,₹7942,72% off,+ ₹261 taxes & fees · per room per night,No,Living Room Reception Toiletries available ...,No


In [123]:
df.tail()

,Unnamed: 0,Hotel_Name,Location,Ratings,Final_Price,Original_Price,Discount,Taxes,Wizard_Member,Amenities,Company_Serviced
1786,1786,Spot On Sonar Bangla Lodge,"Near Howrah railway station, Howrah AC Market,...",3.2 (62 Ratings)·Fair,₹917,₹3336,68% off,+ ₹131 taxes & fees · per room per night,No,Private entrance Parking facility Reception ...,No
1787,1787,Hotel O Mominpur Guest House,"Braunfield Road Mominpore Kolkata, Kolkata",4.8 (82 Ratings)·Excellent,₹955,₹3733,70% off,+ ₹140 taxes & fees · per room per night,Yes,Reception Free Wifi Geyser + 4 more,No
1788,1788,Hotel O Sigma,"Near Buxarah Road, Howrah, Kolkata",3.6 (19 Ratings)·Good,₹907,₹3286,68% off,+ ₹141 taxes & fees · per room per night,No,Free Wifi Geyser Power backup + 6 more,No
1789,1789,Hotel O by OYO Annapurna Plaza,"Market Street, Kolkata",3.0 (1 Ratings)·Fair,₹1330,₹6066,74% off,+ ₹195 taxes & fees · per room per night,No,AC TV,No
1790,1790,Hotel O by OYO 4 Friends Hotel Unit : II,"New Town, Kolkata",3.8 (3 Ratings)·Good,₹974,₹3985,71% off,+ ₹142 taxes & fees · per room per night,No,Elevator Free Wifi Geyser + 3 more,No


In [124]:
hotel_type_pattern = re.compile(
    r'^(super townhouse|townhouse|hotel o|collection o|capital o|palette|flagship|oyo)\b',
    re.IGNORECASE
)

In [125]:
def extract_hotel_type(name):
    if pd.isna(name):
        return np.nan
    
    match = hotel_type_pattern.search(name)
    return match.group(1).title() if match else "Other"


In [126]:
df["Hotel_Type"]=df["Hotel_Name"].apply(extract_hotel_type)

In [127]:
# remove hotel type from start
remove_type_pattern = re.compile(
    r'^(super townhouse|townhouse|hotel o|collection o|capital o|palette|flagship|oyo)\s+',
    re.IGNORECASE
)

# remove ONLY "by OYO" / "OYO" / "OAK"
branding_pattern = re.compile(
    r'\b(by\s+oyo|oyo|oak)\b',
    re.IGNORECASE
)

# remove "Formerly ..." and everything after
formerly_pattern = re.compile(
    r'\bformerly\b.*',
    re.IGNORECASE
)


In [128]:
def extract_hotel_name(name):
    if pd.isna(name):
        return np.nan

    clean = name
    clean = remove_type_pattern.sub('', clean)
    clean = branding_pattern.sub('', clean)
    clean = formerly_pattern.sub('', clean)

    # normalize spaces
    clean = re.sub(r'\s+', ' ', clean).strip()

    return clean


In [129]:
df["Clean_Hotel_Name"]=df["Hotel_Name"].apply(extract_hotel_name)

In [130]:
df.head()

,Unnamed: 0,Hotel_Name,Location,Ratings,Final_Price,Original_Price,Discount,Taxes,Wizard_Member,Amenities,Company_Serviced,Hotel_Type,Clean_Hotel_Name
0,0,Townhouse Narayanaguda Metro Station Formerly ...,"Near Old MLA Quarters, Himayathnagar, Hyderabad",4.3 (774 Ratings)·Very Good,₹1730,₹7095,71% off,+ ₹257 taxes & fees · per room per night,Yes,Parking facility Dining area Free Wifi + 14...,Yes,Townhouse,Narayanaguda Metro Station
1,1,Townhouse Kothapet Formerly Surabhi Elite,"Kothapet, Hyderabad",4.5 (320 Ratings)·Excellent,₹1363,₹7695,79% off,+ ₹239 taxes & fees · per room per night,Yes,Free Wifi Power backup Parking + 10 more,Yes,Townhouse,Kothapet
2,2,Townhouse Oak Madhapur Nera Regency,"Madhapur, Hyderabad",4.3 (1793 Ratings)·Very Good,₹1605,₹6498,72% off,+ ₹169 taxes & fees · per room per night,Yes,Dining area Power backup Parking + 29 more,Yes,Townhouse,Madhapur Nera Regency
3,3,Hotel O by OYO Saraswathi Residency Near SR Na...,"Near Nalanda School, S.R. Nagar, Hyderabad",4.3 (6 Ratings)·Very Good,₹1332,₹5381,71% off,+ ₹175 taxes & fees · per room per night,No,Elevator Free Wifi Geyser + 8 more,No,Hotel O,Saraswathi Residency Near SR Nagar Metro Station
4,4,Super Townhouse OAK Hotel Belsons Taj Mahal,"Near Swapnalok Complex, Patny Centre, Hyderabad",4.6 (82 Ratings)·Excellent,₹1963,₹7942,72% off,+ ₹261 taxes & fees · per room per night,No,Living Room Reception Toiletries available ...,No,Super Townhouse,Hotel Belsons Taj Mahal


In [131]:
df['City']=df["Location"].apply(
    lambda x: re.findall(
        r'(Hyderabad|Bangalore|Bengaluru|Chennai|Mumbai|Delhi|Kolkata|Pune)$',
        x
    )[0]
    if isinstance(x, str) and re.findall(
        r'(Hyderabad|Bangalore|Bengaluru|Chennai|Mumbai|Delhi|Kolkata|Pune)$',
        x
    )
    else np.nan
)


In [132]:
df.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1791 entries, 0 to 1790
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Unnamed: 0        1791 non-null   int64 
 1   Hotel_Name        1791 non-null   object
 2   Location          1791 non-null   object
 3   Ratings           1791 non-null   object
 4   Final_Price       1791 non-null   object
 5   Original_Price    1791 non-null   object
 6   Discount          1791 non-null   object
 7   Taxes             1791 non-null   object
 8   Wizard_Member     1791 non-null   object
 9   Amenities         1791 non-null   object
 10  Company_Serviced  1791 non-null   object
 11  Hotel_Type        1791 non-null   object
 12  Clean_Hotel_Name  1791 non-null   object
 13  City              1617 non-null   object
dtypes: int64(1), object(13)
memory usage: 196.0+ KB


In [133]:
df['Rating']=df['Ratings'].apply(lambda x:re.findall(r'[\d\.]+',x)[0] if re.findall(r'[\d\.]+',x) else np.nan)

In [134]:
df['Number of Ratings']=df['Ratings'].apply(lambda x:re.findall(r'[\d\.]+',x)[1] if re.findall(r'[\d\.]+',x) else np.nan) 

In [135]:
df["Rating_Category"] = df["Ratings"].apply(
    lambda x: re.findall(r'([A-Za-z ]+)$', x)[0].strip()
    if isinstance(x, str) and re.findall(r'([A-Za-z ]+)$', x)
    else None)

In [136]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1791 entries, 0 to 1790
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Unnamed: 0         1791 non-null   int64 
 1   Hotel_Name         1791 non-null   object
 2   Location           1791 non-null   object
 3   Ratings            1791 non-null   object
 4   Final_Price        1791 non-null   object
 5   Original_Price     1791 non-null   object
 6   Discount           1791 non-null   object
 7   Taxes              1791 non-null   object
 8   Wizard_Member      1791 non-null   object
 9   Amenities          1791 non-null   object
 10  Company_Serviced   1791 non-null   object
 11  Hotel_Type         1791 non-null   object
 12  Clean_Hotel_Name   1791 non-null   object
 13  City               1617 non-null   object
 14  Rating             1707 non-null   object
 15  Number of Ratings  1707 non-null   object
 16  Rating_Category    1791 non-null   object


In [137]:
df['Final_Price']=df['Final_Price'].apply(lambda x :re.findall(r'[\d]+',x)[0])

In [138]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1791 entries, 0 to 1790
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Unnamed: 0         1791 non-null   int64 
 1   Hotel_Name         1791 non-null   object
 2   Location           1791 non-null   object
 3   Ratings            1791 non-null   object
 4   Final_Price        1791 non-null   object
 5   Original_Price     1791 non-null   object
 6   Discount           1791 non-null   object
 7   Taxes              1791 non-null   object
 8   Wizard_Member      1791 non-null   object
 9   Amenities          1791 non-null   object
 10  Company_Serviced   1791 non-null   object
 11  Hotel_Type         1791 non-null   object
 12  Clean_Hotel_Name   1791 non-null   object
 13  City               1617 non-null   object
 14  Rating             1707 non-null   object
 15  Number of Ratings  1707 non-null   object
 16  Rating_Category    1791 non-null   object


In [139]:
df['Original_Price']=df['Original_Price'].apply(lambda x:re.findall(r'[\d]+',x)[0])

In [140]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1791 entries, 0 to 1790
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Unnamed: 0         1791 non-null   int64 
 1   Hotel_Name         1791 non-null   object
 2   Location           1791 non-null   object
 3   Ratings            1791 non-null   object
 4   Final_Price        1791 non-null   object
 5   Original_Price     1791 non-null   object
 6   Discount           1791 non-null   object
 7   Taxes              1791 non-null   object
 8   Wizard_Member      1791 non-null   object
 9   Amenities          1791 non-null   object
 10  Company_Serviced   1791 non-null   object
 11  Hotel_Type         1791 non-null   object
 12  Clean_Hotel_Name   1791 non-null   object
 13  City               1617 non-null   object
 14  Rating             1707 non-null   object
 15  Number of Ratings  1707 non-null   object
 16  Rating_Category    1791 non-null   object


In [141]:
df['Discount']=df['Discount'].apply(lambda x:re.findall(r'[\d]+',x)[0])

In [142]:
df['Discount']

0       71
1       79
2       72
3       71
4       72
        ..
1786    68
1787    70
1788    68
1789    74
1790    71
Name: Discount, Length: 1791, dtype: object

In [143]:
df['Tax_Amount']=df['Taxes'].apply(lambda x: re.findall(r'[\d]+',x)[0])

In [144]:
df["Amenity_Count"] = df["Amenities"].apply(
    lambda x: int(re.findall(r'\+\s*(\d+)', x)[0]) + len(x.split('+')[0].split())
    if isinstance(x, str) and re.findall(r'\+\s*(\d+)', x)
    else len(x.split()) if isinstance(x, str)
    else None
)


In [145]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1791 entries, 0 to 1790
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Unnamed: 0         1791 non-null   int64 
 1   Hotel_Name         1791 non-null   object
 2   Location           1791 non-null   object
 3   Ratings            1791 non-null   object
 4   Final_Price        1791 non-null   object
 5   Original_Price     1791 non-null   object
 6   Discount           1791 non-null   object
 7   Taxes              1791 non-null   object
 8   Wizard_Member      1791 non-null   object
 9   Amenities          1791 non-null   object
 10  Company_Serviced   1791 non-null   object
 11  Hotel_Type         1791 non-null   object
 12  Clean_Hotel_Name   1791 non-null   object
 13  City               1617 non-null   object
 14  Rating             1707 non-null   object
 15  Number of Ratings  1707 non-null   object
 16  Rating_Category    1791 non-null   object


In [146]:
df = df.drop(
    columns=["Hotel_Name", "Location", "Ratings"]
)

In [147]:
df.shape

(1791, 16)

In [148]:
df

,Unnamed: 0,Final_Price,Original_Price,Discount,Taxes,Wizard_Member,Amenities,Company_Serviced,Hotel_Type,Clean_Hotel_Name,City,Rating,Number of Ratings,Rating_Category,Tax_Amount,Amenity_Count
0,0,1730,7095,71,+ ₹257 taxes & fees · per room per night,Yes,Parking facility Dining area Free Wifi + 14...,Yes,Townhouse,Narayanaguda Metro Station,Hyderabad,4.3,774,Very Good,257,20
1,1,1363,7695,79,+ ₹239 taxes & fees · per room per night,Yes,Free Wifi Power backup Parking + 10 more,Yes,Townhouse,Kothapet,Hyderabad,4.5,320,Excellent,239,15
2,2,1605,6498,72,+ ₹169 taxes & fees · per room per night,Yes,Dining area Power backup Parking + 29 more,Yes,Townhouse,Madhapur Nera Regency,Hyderabad,4.3,1793,Very Good,169,34
3,3,1332,5381,71,+ ₹175 taxes & fees · per room per night,No,Elevator Free Wifi Geyser + 8 more,No,Hotel O,Saraswathi Residency Near SR Nagar Metro Station,Hyderabad,4.3,6,Very Good,175,12
4,4,1963,7942,72,+ ₹261 taxes & fees · per room per night,No,Living Room Reception Toiletries available ...,No,Super Townhouse,Hotel Belsons Taj Mahal,Hyderabad,4.6,82,Excellent,261,49
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1786,1786,917,3336,68,+ ₹131 taxes & fees · per room per night,No,Private entrance Parking facility Reception ...,No,Other,Spot On Sonar Bangla Lodge,Kolkata,3.2,62,Fair,131,15
1787,1787,955,3733,70,+ ₹140 taxes & fees · per room per night,Yes,Reception Free Wifi Geyser + 4 more,No,Hotel O,Mominpur Guest House,Kolkata,4.8,82,Excellent,140,8
1788,1788,907,3286,68,+ ₹141 taxes & fees · per room per night,No,Free Wifi Geyser Power backup + 6 more,No,Hotel O,Sigma,Kolkata,3.6,19,Good,141,11
1789,1789,1330,6066,74,+ ₹195 taxes & fees · per room per night,No,AC TV,No,Hotel O,Annapurna Plaza,Kolkata,3.0,1,Fair,195,2


In [149]:
df.drop(columns='Taxes',inplace=True)

In [150]:
df.head()

,Unnamed: 0,Final_Price,Original_Price,Discount,Wizard_Member,Amenities,Company_Serviced,Hotel_Type,Clean_Hotel_Name,City,Rating,Number of Ratings,Rating_Category,Tax_Amount,Amenity_Count
0,0,1730,7095,71,Yes,Parking facility Dining area Free Wifi + 14...,Yes,Townhouse,Narayanaguda Metro Station,Hyderabad,4.3,774,Very Good,257,20
1,1,1363,7695,79,Yes,Free Wifi Power backup Parking + 10 more,Yes,Townhouse,Kothapet,Hyderabad,4.5,320,Excellent,239,15
2,2,1605,6498,72,Yes,Dining area Power backup Parking + 29 more,Yes,Townhouse,Madhapur Nera Regency,Hyderabad,4.3,1793,Very Good,169,34
3,3,1332,5381,71,No,Elevator Free Wifi Geyser + 8 more,No,Hotel O,Saraswathi Residency Near SR Nagar Metro Station,Hyderabad,4.3,6,Very Good,175,12
4,4,1963,7942,72,No,Living Room Reception Toiletries available ...,No,Super Townhouse,Hotel Belsons Taj Mahal,Hyderabad,4.6,82,Excellent,261,49


In [151]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1791 entries, 0 to 1790
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Unnamed: 0         1791 non-null   int64 
 1   Final_Price        1791 non-null   object
 2   Original_Price     1791 non-null   object
 3   Discount           1791 non-null   object
 4   Wizard_Member      1791 non-null   object
 5   Amenities          1791 non-null   object
 6   Company_Serviced   1791 non-null   object
 7   Hotel_Type         1791 non-null   object
 8   Clean_Hotel_Name   1791 non-null   object
 9   City               1617 non-null   object
 10  Rating             1707 non-null   object
 11  Number of Ratings  1707 non-null   object
 12  Rating_Category    1791 non-null   object
 13  Tax_Amount         1791 non-null   object
 14  Amenity_Count      1791 non-null   int64 
dtypes: int64(2), object(13)
memory usage: 210.0+ KB


In [152]:
df["City"].fillna(df["City"].mode()[0], inplace=True)


In [153]:
df['Rating'].fillna(df['Rating'].mode()[0],inplace=True)

In [154]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1791 entries, 0 to 1790
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Unnamed: 0         1791 non-null   int64 
 1   Final_Price        1791 non-null   object
 2   Original_Price     1791 non-null   object
 3   Discount           1791 non-null   object
 4   Wizard_Member      1791 non-null   object
 5   Amenities          1791 non-null   object
 6   Company_Serviced   1791 non-null   object
 7   Hotel_Type         1791 non-null   object
 8   Clean_Hotel_Name   1791 non-null   object
 9   City               1791 non-null   object
 10  Rating             1791 non-null   object
 11  Number of Ratings  1707 non-null   object
 12  Rating_Category    1791 non-null   object
 13  Tax_Amount         1791 non-null   object
 14  Amenity_Count      1791 non-null   int64 
dtypes: int64(2), object(13)
memory usage: 210.0+ KB


In [155]:
df["Number of Ratings"] = df["Number of Ratings"].apply(
    lambda x: int(re.findall(r'\d+', x)[0])
    if isinstance(x, str) and re.findall(r'\d+', x)
    else None
)


In [156]:
df['Number of Ratings'].fillna(df['Number of Ratings'].median(),inplace=True)

In [157]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1791 entries, 0 to 1790
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         1791 non-null   int64  
 1   Final_Price        1791 non-null   object 
 2   Original_Price     1791 non-null   object 
 3   Discount           1791 non-null   object 
 4   Wizard_Member      1791 non-null   object 
 5   Amenities          1791 non-null   object 
 6   Company_Serviced   1791 non-null   object 
 7   Hotel_Type         1791 non-null   object 
 8   Clean_Hotel_Name   1791 non-null   object 
 9   City               1791 non-null   object 
 10  Rating             1791 non-null   object 
 11  Number of Ratings  1791 non-null   float64
 12  Rating_Category    1791 non-null   object 
 13  Tax_Amount         1791 non-null   object 
 14  Amenity_Count      1791 non-null   int64  
dtypes: float64(1), int64(2), object(12)
memory usage: 210.0+ KB


In [168]:
df['Final_Price']=df['Final_Price'].astype(np.int64)
df['Original_Price']=df['Original_Price'].astype(np.int64)
df['Discount']=df['Discount'].astype(np.int64)
df['Rating']=df['Rating'].astype(np.float64)
df['Number of Ratings']=df['Number of Ratings'].astype(np.int64)
df['Tax_Amount']=df['Tax_Amount'].astype(np.int64)

In [162]:
df.drop(columns={'Unnamed: 0'},inplace=True)

In [163]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1791 entries, 0 to 1790
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Final_Price        1791 non-null   object 
 1   Original_Price     1791 non-null   int64  
 2   Discount           1791 non-null   int64  
 3   Wizard_Member      1791 non-null   object 
 4   Amenities          1791 non-null   object 
 5   Company_Serviced   1791 non-null   object 
 6   Hotel_Type         1791 non-null   object 
 7   Clean_Hotel_Name   1791 non-null   object 
 8   City               1791 non-null   object 
 9   Rating             1791 non-null   float64
 10  Number of Ratings  1791 non-null   int64  
 11  Rating_Category    1791 non-null   object 
 12  Tax_Amount         1791 non-null   int64  
 13  Amenity_Count      1791 non-null   int64  
dtypes: float64(1), int64(5), object(8)
memory usage: 196.0+ KB


In [164]:
df.head()

,Final_Price,Original_Price,Discount,Wizard_Member,Amenities,Company_Serviced,Hotel_Type,Clean_Hotel_Name,City,Rating,Number of Ratings,Rating_Category,Tax_Amount,Amenity_Count
0,1730,7095,71,Yes,Parking facility Dining area Free Wifi + 14...,Yes,Townhouse,Narayanaguda Metro Station,Hyderabad,4.3,774,Very Good,257,20
1,1363,7695,79,Yes,Free Wifi Power backup Parking + 10 more,Yes,Townhouse,Kothapet,Hyderabad,4.5,320,Excellent,239,15
2,1605,6498,72,Yes,Dining area Power backup Parking + 29 more,Yes,Townhouse,Madhapur Nera Regency,Hyderabad,4.3,1793,Very Good,169,34
3,1332,5381,71,No,Elevator Free Wifi Geyser + 8 more,No,Hotel O,Saraswathi Residency Near SR Nagar Metro Station,Hyderabad,4.3,6,Very Good,175,12
4,1963,7942,72,No,Living Room Reception Toiletries available ...,No,Super Townhouse,Hotel Belsons Taj Mahal,Hyderabad,4.6,82,Excellent,261,49


In [169]:
df['Final_Price'].min()

417

In [170]:
df['Final_Price'].max()

4883

In [166]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1791 entries, 0 to 1790
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Final_Price        1791 non-null   object 
 1   Original_Price     1791 non-null   int64  
 2   Discount           1791 non-null   int64  
 3   Wizard_Member      1791 non-null   object 
 4   Amenities          1791 non-null   object 
 5   Company_Serviced   1791 non-null   object 
 6   Hotel_Type         1791 non-null   object 
 7   Clean_Hotel_Name   1791 non-null   object 
 8   City               1791 non-null   object 
 9   Rating             1791 non-null   float64
 10  Number of Ratings  1791 non-null   int64  
 11  Rating_Category    1791 non-null   object 
 12  Tax_Amount         1791 non-null   int64  
 13  Amenity_Count      1791 non-null   int64  
dtypes: float64(1), int64(5), object(8)
memory usage: 196.0+ KB


In [171]:
def price_category(price):
    if price <= 1000:
        return "Budget"
    elif price <= 2000:
        return "Mid"
    elif price <= 3500:
        return "Premium"
    else:
        return "Luxury"

df["Price_Category"] = df["Final_Price"].apply(price_category)


In [172]:
df["Effective_Price"] = df["Final_Price"] + df["Tax_Amount"]


In [173]:
df["Discount_Amount"] = df["Original_Price"] - df["Final_Price"]


In [174]:
df["Popularity_Score"] = df["Rating"] * df["Number of Ratings"]


In [175]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1791 entries, 0 to 1790
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Final_Price        1791 non-null   int64  
 1   Original_Price     1791 non-null   int64  
 2   Discount           1791 non-null   int64  
 3   Wizard_Member      1791 non-null   object 
 4   Amenities          1791 non-null   object 
 5   Company_Serviced   1791 non-null   object 
 6   Hotel_Type         1791 non-null   object 
 7   Clean_Hotel_Name   1791 non-null   object 
 8   City               1791 non-null   object 
 9   Rating             1791 non-null   float64
 10  Number of Ratings  1791 non-null   int64  
 11  Rating_Category    1791 non-null   object 
 12  Tax_Amount         1791 non-null   int64  
 13  Amenity_Count      1791 non-null   int64  
 14  Price_Category     1791 non-null   object 
 15  Effective_Price    1791 non-null   int64  
 16  Discount_Amount    1791 

In [176]:
df.to_csv('cleanedData_oyo_rooms.csv',mode='x')